# 100k Evaluation — In-Distribution Gate, Then OOD (Pre-Registered)

**This notebook enforces the Log's outcome-mapping protocol mechanically:**
it will not print or interpret any OOD table unless the in-distribution
gate passes first. This is not a formatting choice — it is the fix for
the exact gap identified in Experiment 28 (50k campaign): OOD comparisons
were drawn with no matched-steps significance test and no prior
convergence check.

**Pre-registered in-distribution gate criteria (fixed now, before any
100k number is seen):**
1. **Convergence relative to 50k.** Each condition's 100k MAE on Lorenz
   must be <= 50% of its own 50k value (baseline: 0.275 -> target
   <= 0.138; ablation: 0.617 -> target <= 0.309). This is a real-
   progress check, not a comparison between conditions — it fires
   before the conditions are compared to each other at all.
2. **Direct paired test.** Wilcoxon signed-rank, ablation vs. baseline
   MAE, per-window, n=20, on Lorenz and on each held-out system
   separately. This is the test the 50k campaign never ran.
3. **Gate verdict:** PASS only if criterion 1 holds for BOTH conditions.
   Criterion 2 is reported regardless (it IS the finding, not a gate on
   reporting it) but its result is only trusted for downstream OOD
   interpretation if criterion 1 passed.

**Outcome mapping (from the Log, restated here so it is fixed before
results are seen):**
- Convergence gate passes AND ablation is significantly worse than
  baseline in-distribution AND the Burgers-worse / periodic-neutral OOD
  pattern from Experiment 28 persists -> Koopman lifting hypothesis
  rises to medium confidence.
- Convergence gate passes AND in-distribution ablation/baseline gap
  closes (not significant, or ablation competitive) -> 50k OOD pattern
  was likely a training-instability artifact; lifting hypothesis not
  supported.
- Convergence gate FAILS for either condition -> STOP. No OOD claim is
  drawn from this checkpoint pair. Report which condition failed to
  converge and by how much; do not lower the threshold post hoc.

**Two data-source TODOs, deliberately left unfilled rather than guessed.**
This notebook does not know which two skew40 systems are held-out from
training, or your exact Burgers/Van der Pol/Duffing/Harmonic simulators
used to produce the Koopman ablation CSVs — inventing those would risk
silently diverging from what actually backs the existing 50k numbers.
Both are marked `# TODO` below with the exact interface required.

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

N_WINDOWS   = 20
CONTEXT_LEN = 512
OUT_NPZ_DIR = './eval100k_raw_predictions'
os.makedirs(OUT_NPZ_DIR, exist_ok=True)

# Convergence targets, fixed from the 50k campaign (Log Experiment 28 /
# the anecdotal in-distribution figures). Do not adjust after seeing
# 100k results.
MAE_50K_BASELINE = 0.275
MAE_50K_ABLATION = 0.617
CONV_FACTOR      = 0.50   # 100k must be <= this fraction of the 50k value

## Load both 100k checkpoints

Both were saved with `model.save_pretrained()`, so `PatchTSTPipeline
.from_pretrained` should accept the local directory directly — this
reuses the exact inference path (windowed autoregression, generation
config) rather than reimplementing it, avoiding a second source of
divergence from the training notebooks.

In [ ]:
import sys
import os, json

# =====================================================================
# Robust checkpoint locator -- reuses the pattern already proven during
# the 50k campaign (two prior path-mismatch failures: Kaggle mount path
# including the dataset owner slug, and an extra nesting level from a
# zip upload). Searches under /kaggle/input rather than assuming a
# fixed path, so it doesn't repeat either failure a third time.
#
# ADJUST: set this to whatever you name the Kaggle dataset once you
# upload both 100k checkpoints (as sibling folders, e.g.
# <dataset>/baseline/... and <dataset>/koopman_ablation/...).
# =====================================================================
DATASET_NAME_HINT = 'panda-100k-checkpoints'  # ADJUST to your actual dataset slug/name

print('Contents of /kaggle/input:')
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.replace('/kaggle/input', '').count(os.sep)
    if depth <= 2:
        print('  ' * depth + os.path.basename(root) + '/')

_root_matches = [os.path.join(r, d) for r, dirs, _ in os.walk('/kaggle/input')
                  for d in dirs if DATASET_NAME_HINT in d]
if not _root_matches:
    raise FileNotFoundError(
        f"Could not find a folder containing '{DATASET_NAME_HINT}' anywhere "
        f"under /kaggle/input. See the tree printed above and set "
        f"DATASET_NAME_HINT to match your actual upload."
    )
DATASET_ROOT = _root_matches[0]
print(f'\nDATASET_ROOT resolved to: {DATASET_ROOT}')


def find_checkpoint_dir(dataset_root, expected_use_dyn):
    # Searches for a directory containing both model.safetensors and
    # training_info.json whose use_dynamics_embedding matches
    # expected_use_dyn. Errors loudly on zero or multiple matches
    # rather than guessing.
    candidates = []
    for dirpath, dirnames, filenames in os.walk(dataset_root):
        if 'training_info.json' in filenames and 'model.safetensors' in filenames:
            try:
                with open(os.path.join(dirpath, 'training_info.json')) as f:
                    info = json.load(f)
                if info.get('use_dynamics_embedding') == expected_use_dyn:
                    candidates.append(dirpath)
            except Exception:
                pass
    candidates = sorted(set(candidates))
    if len(candidates) == 0:
        print(f'\nFull directory tree under {dataset_root}:')
        for dirpath, dirnames, filenames in os.walk(dataset_root):
            depth = dirpath.replace(dataset_root, '').count(os.sep)
            print('  ' * depth + os.path.basename(dirpath) + '/')
            for fn in filenames:
                print('  ' * (depth + 1) + fn)
        raise FileNotFoundError(
            f'No checkpoint found with use_dynamics_embedding={expected_use_dyn} '
            f'under {dataset_root}. Tree printed above -- locate manually if the '
            f'search heuristic missed it.'
        )
    elif len(candidates) > 1:
        raise RuntimeError(
            f'Multiple matching checkpoints found for use_dynamics_embedding='
            f'{expected_use_dyn}: {candidates}. Ambiguous -- fix the dataset '
            f'layout so only one exists.'
        )
    return candidates[0]


BASELINE_100K_DIR = find_checkpoint_dir(DATASET_ROOT, expected_use_dyn=True)
ABLATION_100K_DIR = find_checkpoint_dir(DATASET_ROOT, expected_use_dyn=False)
print(f'\nBASELINE_100K_DIR resolved to: {BASELINE_100K_DIR}')
print(f'ABLATION_100K_DIR resolved to: {ABLATION_100K_DIR}')

for _dir, _name in [(BASELINE_100K_DIR, 'baseline'), (ABLATION_100K_DIR, 'ablation')]:
    with open(os.path.join(_dir, 'training_info.json')) as f:
        _info = json.load(f)
    print(f'{_name}: total_steps={_info.get("total_steps")}, '
          f'use_dynamics_embedding={_info.get("use_dynamics_embedding")}')

sys.path.insert(0, '/kaggle/working/panda')  # adjust if your Kaggle session clones it elsewhere
from panda.patchtst.pipeline import PatchTSTPipeline

pipe_baseline = PatchTSTPipeline.from_pretrained(
    mode='predict', pretrain_path=BASELINE_100K_DIR, device_map=device,
)
pipe_ablation = PatchTSTPipeline.from_pretrained(
    mode='predict', pretrain_path=ABLATION_100K_DIR, device_map=device,
)
print('Both 100k checkpoints loaded.')


## Harness — mirrors `panda_forecast` from `new_experiments.ipynb`,
parametrised by which pipeline to call (the original was hardcoded to
one global `panda_model`; this is the only structural change, kept
otherwise byte-identical to preserve comparability with all prior log
results.)

In [ ]:
def instance_norm_window(x_CT):
    mu  = x_CT.mean(axis=1, keepdims=True)
    std = x_CT.std( axis=1, keepdims=True) + 1e-8
    return (x_CT - mu) / std, mu, std

def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def panda_forecast_with(pipe, context_np, horizon):
    """Verbatim panda_forecast logic, parametrised by pipeline object."""
    TRAIN_H   = 128
    remaining = horizon
    ctx       = context_np.copy()
    preds     = []
    while remaining > 0:
        h         = min(TRAIN_H, remaining)
        context_t = torch.tensor(ctx.T, dtype=torch.float32)
        with torch.no_grad():
            pred = pipe.predict(
                context_t, h,
                limit_prediction_length=False,
                sliding_context=True,
            )
        p = pred.squeeze().cpu().numpy()
        if p.ndim == 1:
            p = p[:, None]
        if p.shape[0] != context_np.shape[0]:
            p = p.T
        preds.append(p[:, :h])
        ctx       = np.concatenate([ctx[:, h:], p[:, :h]], axis=1)
        remaining -= h
    return np.concatenate(preds, axis=1)

def paired_evaluate(data_CT, horizon, label, n_windows=N_WINDOWS, save_npz=True):
    """Directly compares baseline vs. ablation on the SAME windows
    (paired), which the 50k campaign never did. Saves raw predictions."""
    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f'  [SKIP] {label}: T={T} too short')
        return None

    starts = np.linspace(0, max_start, n_windows, dtype=int)
    mae_base, mae_abl = [], []
    preds_base, preds_abl, tgts = [], [], []

    for s in starts:
        ctx_raw           = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw           = data_CT[:, s + CONTEXT_LEN : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm          = (tgt_raw - mu) / std
        pb = panda_forecast_with(pipe_baseline, ctx_norm, horizon)
        pa = panda_forecast_with(pipe_ablation, ctx_norm, horizon)
        mae_base.append(mae(tgt_norm, pb))
        mae_abl.append(mae(tgt_norm, pa))
        preds_base.append(pb); preds_abl.append(pa); tgts.append(tgt_norm)

    if save_npz:
        np.savez_compressed(
            f'{OUT_NPZ_DIR}/{label}.npz',
            preds_baseline=np.array(preds_base), preds_ablation=np.array(preds_abl),
            targets=np.array(tgts), starts=starts,
        )

    diff = np.array(mae_abl) - np.array(mae_base)  # >0 means ablation worse
    try:
        _, p_greater = wilcoxon(diff, alternative='greater') if np.any(diff != 0) else (0, 1.0)
        _, p_less    = wilcoxon(diff, alternative='less')    if np.any(diff != 0) else (0, 1.0)
    except Exception:
        p_greater = p_less = np.nan

    result = {
        'label': label, 'horizon': horizon, 'n_windows': n_windows,
        'baseline_mae': np.median(mae_base), 'baseline_iqr':
            np.percentile(mae_base,75) - np.percentile(mae_base,25),
        'ablation_mae': np.median(mae_abl), 'ablation_iqr':
            np.percentile(mae_abl,75) - np.percentile(mae_abl,25),
        'ablation_minus_baseline': np.median(mae_abl) - np.median(mae_base),
        'p_ablation_worse': p_greater, 'p_ablation_better': p_less,
    }
    sig = ' *ABL WORSE' if p_greater < 0.05 else (' *ABL BETTER' if p_less < 0.05 else '')
    print(f'  {label:32s} H={horizon:4d}  base={np.median(mae_base):.4f}  '
          f'abl={np.median(mae_abl):.4f}  \u0394={result["ablation_minus_baseline"]:+.4f}  '
          f'p(worse)={p_greater:.3f} p(better)={p_less:.3f}{sig}')
    return result

print('Harness defined.')

## STAGE 1 — In-Distribution Gate (Lorenz; run this before anything else)

In [ ]:
def simulate_lorenz(n=5000, dt=0.01, sigma=10, rho=28, beta=8/3):
    """Verbatim from the TDA gate notebooks."""
    x, y, z = 0.1, 0.0, 0.0
    xs, ys, zs = [x], [y], [z]
    for _ in range(n - 1):
        k1x = sigma * (y - x); k1y = x * (rho - z) - y; k1z = x * y - beta * z
        k2x = sigma * ((y + dt/2*k1y) - (x + dt/2*k1x))
        k2y = (x + dt/2*k1x) * (rho - (z + dt/2*k1z)) - (y + dt/2*k1y)
        k2z = (x + dt/2*k1x) * (y + dt/2*k1y) - beta * (z + dt/2*k1z)
        k3x = sigma * ((y + dt/2*k2y) - (x + dt/2*k2x))
        k3y = (x + dt/2*k2x) * (rho - (z + dt/2*k2z)) - (y + dt/2*k2y)
        k3z = (x + dt/2*k2x) * (y + dt/2*k2y) - beta * (z + dt/2*k2z)
        k4x = sigma * ((y + dt*k3y) - (x + dt*k3x))
        k4y = (x + dt*k3x) * (rho - (z + dt*k3z)) - (y + dt*k3y)
        k4z = (x + dt*k3x) * (y + dt*k3y) - beta * (z + dt*k3z)
        x += dt/6*(k1x+2*k2x+2*k3x+k4x)
        y += dt/6*(k1y+2*k2y+2*k3y+k4y)
        z += dt/6*(k1z+2*k2z+2*k3z+k4z)
        xs.append(x); ys.append(y); zs.append(z)
    return np.array([xs, ys, zs]).T

lorenz_traj = simulate_lorenz(n=5000)[500:3500]      # discard transient
lorenz_CT   = lorenz_traj.T                          # (3, 3000)
print(f'Lorenz reference trajectory: {lorenz_CT.shape}')

lorenz_result = paired_evaluate(lorenz_CT, horizon=96, label='gate_lorenz_H96')

## TODO 1 — Two held-out skew40 systems

The gate should not rest on Lorenz alone: it is one trajectory and may
not represent the training distribution's convergence broadly. Fill in
TWO systems confirmed NOT present in the skew40 training split (check
against your skew40 loading metadata, e.g. `target._np_shape` /
whatever field records system identity, per the project's established
loading approach). Do not substitute systems already known to be in
skew40 — that would test memorisation, not generalisation-within-
distribution.

Fill `held_out_trajectories` below as a dict of {name: (C,T) array},
generated however your skew40/dysts pipeline does it, matching the
Lorenz shape convention above (channels x time).

In [ ]:
held_out_trajectories = {
    # 'system_name_1': traj_CT_1,   # TODO: fill in
    # 'system_name_2': traj_CT_2,   # TODO: fill in
}

gate_results = [lorenz_result] if lorenz_result else []
for name, traj_CT in held_out_trajectories.items():
    r = paired_evaluate(traj_CT, horizon=96, label=f'gate_{name}_H96')
    if r:
        gate_results.append(r)

if not held_out_trajectories:
    print('\nWARNING: held_out_trajectories is empty. Proceeding on Lorenz '
          'alone weakens the gate — fill in TODO 1 before treating any '
          'gate verdict below as final.')

In [ ]:
# Optional helper (not required to run the gate): lists unique source
# systems in the loaded skew40 corpus, so you can identify 2 systems
# confirmed ABSENT from this list as held-out candidates for TODO 1
# later. Requires hf_dataset (the skew40 HF dataset object) already
# loaded in scope, per your Cell 2 pattern from the training notebook.
try:
    unique_sources = sorted(set(hf_dataset['_source_directory']))
    print(f'{len(unique_sources)} unique source systems in the loaded skew40 split:')
    for s in unique_sources:
        print(f'  {s}')
    print('\nPick 2 systems NOT in this list (verified absent from skew40)')
    print('as held-out candidates, then fill held_out_trajectories above.')
except NameError:
    print('hf_dataset not in scope -- load skew40 first if you want this listing.')


## Gate Verdict — mechanically enforced, not a suggestion

In [ ]:
# Convergence check needs the RAW 100k Lorenz MAE, not the paired-eval
# median against a shared window set alone — recompute single-condition
# MAE over the same Lorenz windows for the convergence criterion.
def single_condition_mae(pipe, data_CT, horizon, n_windows=N_WINDOWS):
    C, T = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    starts = np.linspace(0, max_start, n_windows, dtype=int)
    maes = []
    for s in starts:
        ctx_raw = data_CT[:, s:s+CONTEXT_LEN]
        tgt_raw = data_CT[:, s+CONTEXT_LEN:s+CONTEXT_LEN+horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm = (tgt_raw - mu) / std
        p = panda_forecast_with(pipe, ctx_norm, horizon)
        maes.append(mae(tgt_norm, p))
    return float(np.median(maes))

lorenz_mae_baseline_100k = single_condition_mae(pipe_baseline, lorenz_CT, 96)
lorenz_mae_ablation_100k = single_condition_mae(pipe_ablation, lorenz_CT, 96)

conv_base = lorenz_mae_baseline_100k <= CONV_FACTOR * MAE_50K_BASELINE
conv_abl  = lorenz_mae_ablation_100k <= CONV_FACTOR * MAE_50K_ABLATION

print(f'Baseline Lorenz MAE: 50k={MAE_50K_BASELINE:.4f} -> 100k='
      f'{lorenz_mae_baseline_100k:.4f}  (target <= {CONV_FACTOR*MAE_50K_BASELINE:.4f})  '
      f'{"PASS" if conv_base else "FAIL"}')
print(f'Ablation Lorenz MAE: 50k={MAE_50K_ABLATION:.4f} -> 100k='
      f'{lorenz_mae_ablation_100k:.4f}  (target <= {CONV_FACTOR*MAE_50K_ABLATION:.4f})  '
      f'{"PASS" if conv_abl else "FAIL"}')

GATE_PASSED = conv_base and conv_abl
print(f'\n{"="*60}')
print(f'IN-DISTRIBUTION GATE: {"PASS" if GATE_PASSED else "FAIL"}')
print(f'{"="*60}')

if not GATE_PASSED:
    print('\nGATE FAILED. Per the pre-registered protocol, STOP HERE.')
    print('Do not run or interpret the OOD cells below. Report which')
    print('condition failed convergence and by how much; the threshold')
    print('(50% of 50k MAE) is not to be relaxed after seeing this result.')
    failed = []
    if not conv_base: failed.append('baseline')
    if not conv_abl:  failed.append('ablation')
    print(f'Failed condition(s): {failed}')

gate_df = pd.DataFrame(gate_results + [{
    'label': 'convergence_check', 'baseline_mae_100k': lorenz_mae_baseline_100k,
    'ablation_mae_100k': lorenz_mae_ablation_100k,
    'conv_base_pass': conv_base, 'conv_abl_pass': conv_abl,
    'gate_passed': GATE_PASSED,
}])
gate_df.to_csv('gate_results.csv', index=False)
print('\nSaved gate_results.csv')

## STAGE 2 — OOD Evaluation (ONLY proceeds if GATE_PASSED)

### TODO 2 — OOD data loaders
Fill in `load_weather`, `load_burgers_nu1`, `load_vanderpol`,
`load_duffing`, `load_harmonic` using your existing simulators/loaders
from `new_experiments.ipynb` and the Burgers notebook — the same ones
that produced `koopman_ablation_results.csv` and
`koopman_ablation_continuum.csv`, so the 50k-vs-100k comparison stays
apples-to-apples. Not reimplemented here to avoid silently diverging
from what actually backs the existing 50k numbers.

In [ ]:
DATA_DIR = './ts_data'  # adjust if this differs from new_experiments.ipynb's path

def load_ts(path):
    """Verbatim from new_experiments.ipynb."""
    df = pd.read_csv(path)
    df = df.select_dtypes(include=[np.number])
    return df.values.astype(np.float32).T  # (C, T)


def simulate_burgers_stable(T=1000, N_x=128, nu=0.005, seed=SEED):
    """Verbatim from new_experiments.ipynb (Cell 11)."""
    from scipy.fft import fft, ifft, fftfreq
    rng = np.random.default_rng(seed)
    dx  = 2 * np.pi / N_x

    dt_diff   = 0.4 * dx**2 / (2 * nu + 1e-10)
    dt_adv    = 0.4 * dx
    dt        = min(dt_diff, dt_adv, 0.05)
    dt_record = 0.01
    n_sub     = max(1, int(np.ceil(dt_record / dt)))
    dt_act    = dt_record / n_sub

    k       = fftfreq(N_x, d=1.0/N_x).astype(complex)
    dealias = np.abs(k) <= N_x // 3
    L_op    = -nu * k**2

    u0_hat = np.zeros(N_x, dtype=complex)
    for m in range(1, 6):
        amp = rng.standard_normal() + 1j * rng.standard_normal()
        u0_hat[m]       += amp
        u0_hat[N_x - m] += np.conj(amp)
    u0_hat *= dealias

    def rhs_hat(u_hat):
        u_phys = np.real(ifft(u_hat))
        nonlin = fft(0.5 * u_phys**2) * dealias
        return L_op * u_hat - 1j * k * nonlin

    U     = np.zeros((T, N_x), dtype=np.float32)
    u_hat = u0_hat.copy()
    for t in range(T):
        U[t] = np.real(ifft(u_hat)).astype(np.float32)
        for _ in range(n_sub):
            k1    = rhs_hat(u_hat)
            k2    = rhs_hat(u_hat + 0.5*dt_act*k1)
            k3    = rhs_hat(u_hat + 0.5*dt_act*k2)
            k4    = rhs_hat(u_hat +     dt_act*k3)
            u_hat = u_hat + (dt_act/6.0)*(k1+2*k2+2*k3+k4)
            u_hat *= dealias
            if not np.isfinite(u_hat).all():
                print(f'    Diverged at t={t}')
                return U[:t]
    return U


def pca_reduction(U, n_components):
    """Verbatim from new_experiments.ipynb (Cell 11)."""
    from scipy.linalg import svd
    U_c  = U - U.mean(axis=0, keepdims=True)
    n_c  = min(n_components, min(U_c.shape)-1)
    _, _, Vt = svd(U_c, full_matrices=False)
    return (U_c @ Vt[:n_c].T).astype(np.float32)  # (T, n_c)


def simulate_harmonic(n_steps=3000, omega=1.0, seed=SEED):
    """Verbatim from new_experiments.ipynb (Cell 40)."""
    rng = np.random.default_rng(seed)
    dt  = 0.05
    x, v = float(rng.standard_normal()), float(rng.standard_normal())
    traj = []
    for _ in range(n_steps):
        traj.append(x)
        x_new = x + v * dt
        v_new = v - omega**2 * x * dt
        x, v  = x_new, v_new
    return np.array(traj, dtype=np.float32)


def simulate_vanderpol(n_steps=3000, mu=2.0, seed=SEED):
    """Verbatim from new_experiments.ipynb (Cell 40)."""
    from scipy.integrate import solve_ivp
    rng = np.random.default_rng(seed)
    def vdp(t, y):
        return [y[1], mu*(1 - y[0]**2)*y[1] - y[0]]
    ic  = rng.standard_normal(2).tolist()
    sol = solve_ivp(vdp, [0, n_steps*0.05], ic,
                    t_eval=np.linspace(0, n_steps*0.05, n_steps),
                    method='RK45', rtol=1e-8, atol=1e-8)
    return sol.y[0].astype(np.float32)


def simulate_duffing(n_steps=3000, delta=0.3, alpha=-1.0,
                     beta=1.0, gamma=0.37, omega=1.2, seed=SEED):
    """Verbatim from new_experiments.ipynb (Cell 40)."""
    rng = np.random.default_rng(seed)
    dt  = 2*np.pi / omega / 50
    x, v = float(rng.standard_normal()), float(rng.standard_normal())
    traj = []
    t    = 0.0
    for _ in range(n_steps):
        traj.append(x)
        ax    = -delta*v - alpha*x - beta*x**3 + gamma*np.cos(omega*t)
        x_new = x + v*dt
        v_new = v + ax*dt
        x, v, t = x_new, v_new, t+dt
    return np.array(traj, dtype=np.float32)


# -------------------------------------------------------
# TODO 2 loaders, now filled using the functions above.
# -------------------------------------------------------
def load_weather():
    return load_ts(f'{DATA_DIR}/weather.csv')

def load_burgers_nu1():
    # T=1500, N_x=128, PCA to 16 channels: matches Experiment 10/28's
    # established Burgers protocol exactly (new_experiments.ipynb Cells 11-12).
    U = simulate_burgers_stable(T=1500, N_x=128, nu=1.0, seed=SEED)
    pca_series = pca_reduction(U, 16)
    return pca_series.T  # (16, T)

def load_vanderpol():
    series = simulate_vanderpol(n_steps=4000, mu=2.0, seed=SEED)
    return series[500:][None, :]  # discard transient, (1, T)

def load_duffing():
    series = simulate_duffing(n_steps=4000, seed=SEED)
    return series[500:][None, :]  # discard transient, (1, T)

def load_harmonic():
    series = simulate_harmonic(n_steps=4000, omega=1.0, seed=SEED)
    return series[500:][None, :]  # discard transient, (1, T)


OOD_LOADERS = {
    'Weather':     (load_weather,     [96, 192, 336]),
    'Burgers_nu1': (load_burgers_nu1, [96, 192, 336]),
    'VanDerPol':   (load_vanderpol,   [96, 192, 336]),
    'Duffing':     (load_duffing,     [96, 192, 336]),
    'Harmonic':    (load_harmonic,    [96, 192, 336]),
}

print('TODO 2 filled: all five OOD loaders defined, reusing new_experiments.ipynb code verbatim.')


In [ ]:
if GATE_PASSED:
    ood_results = []
    for name, (loader_fn, horizons) in OOD_LOADERS.items():
        try:
            data_CT = loader_fn()
        except NotImplementedError as e:
            print(f'[SKIP] {name}: {e}')
            continue
        print(f'\n=== {name} ===')
        for H in horizons:
            r = paired_evaluate(data_CT, H, label=f'{name}_H{H}')
            if r:
                r['dataset'] = name
                ood_results.append(r)

    if ood_results:
        df_ood = pd.DataFrame(ood_results)
        df_ood.to_csv('ood_100k_results.csv', index=False)
        print('\nSaved ood_100k_results.csv')

        df_ood['ablation_worse'] = df_ood.p_ablation_worse < 0.05
        df_ood['ablation_better'] = df_ood.p_ablation_better < 0.05
        print('\nSummary: significant direction per dataset/horizon')
        print(df_ood[['label', 'ablation_minus_baseline',
                      'ablation_worse', 'ablation_better']].to_string(index=False))
    else:
        print('\nNo OOD datasets loaded — fill in TODO 2 loaders above and rerun this cell.')
else:
    print('OOD evaluation skipped: gate did not pass. See Stage 1 output.')

## Interpretation (only meaningful if GATE_PASSED and TODO 2 is filled)

Apply the pre-registered outcome mapping from the header:
- Ablation significantly worse on Burgers-like datasets AND
  competitive-or-better on periodic datasets (Van der Pol, Duffing,
  Harmonic) AND the in-distribution gate passed -> Koopman lifting
  hypothesis rises to medium confidence, consistent with and now
  properly gated relative to the Experiment 28 pattern.
- Any other pattern -> report exactly what was observed against this
  mapping; do not construct a new post hoc narrative to fit the result.